In [ ]:
# Cell 1: Environment setup and installation
!pip install -q "chronos-forecasting>=2.0" pydantic pyyaml tenacity joblib

import subprocess
import sys

import chronos
import torch

print("=== Kaggle Environment ===")
print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device count:", torch.cuda.device_count())
    print("Device name:", torch.cuda.get_device_name(0))

subprocess.run([sys.executable, "-m", "pip", "show", "chronos-forecasting"])


In [ ]:
# Cell 2: Locate bundle and load terra modules
import glob
import json
import os
import sys
import time

import pandas as pd

# Locate uploaded bundle in /kaggle/input/
bundle_files = glob.glob("/kaggle/input/**/dataset.parquet", recursive=True)
if not bundle_files:
    raise FileNotFoundError("Could not find dataset.parquet in /kaggle/input/.")
bundle_dir = os.path.dirname(bundle_files[0])
print(f"Bundle directory: {bundle_dir}")

# Locate and import terra package
src_init = glob.glob(f"{bundle_dir}/**/terra/__init__.py", recursive=True)
if not src_init:
    zip_files = glob.glob(f"{bundle_dir}/**/terra_src.zip", recursive=True)
    if zip_files:
        import zipfile
        zipfile.ZipFile(zip_files[0]).extractall("/kaggle/working/terra_extracted")
        src_init = glob.glob("/kaggle/working/terra_extracted/**/terra/__init__.py", recursive=True)

if not src_init:
    raise FileNotFoundError("Could not locate terra package in bundle.")

terra_parent = os.path.dirname(os.path.dirname(src_init[0]))
if terra_parent not in sys.path:
    sys.path.insert(0, terra_parent)

from terra.config import TerraConfig, load_yaml
from terra.models.chronos2 import Chronos2Forecaster

cfg = TerraConfig.model_validate(load_yaml(f"{bundle_dir}/site.yaml"))
ds = pd.read_parquet(f"{bundle_dir}/dataset.parquet")
print(f"Dataset loaded: {ds.shape}, index {ds.index[0]} to {ds.index[-1]}")


In [ ]:
# Cell 3: Zero-shot Chronos-2 inference with weather covariates
model_id = "amazon/chronos-2"
device = "cuda" if torch.cuda.is_available() else "cpu"
context_h = 1024
batch_issues = 32
horizon_h = 48

print(f"Initializing Chronos2Forecaster on {device} (model: {model_id})...")
fc = Chronos2Forecaster(model_id=model_id, device=device, context_h=context_h, batch_issues=batch_issues)

import importlib.metadata

try:
    pkg_ver = importlib.metadata.version("chronos-forecasting")
except Exception:
    pkg_ver = getattr(chronos, "__version__", "unknown")

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
out_meta = {
    "model_id": model_id,
    "package_version": pkg_ver,
    "gpu": gpu_name,
    "context_h": context_h,
    "batch_issues": batch_issues,
    "horizon_h": horizon_h,
    "runs": {}
}

out_dir = "/kaggle/working"
os.makedirs(out_dir, exist_ok=True)

for source in ("solar", "wind"):
    cap = cfg.capacity_mw(source)
    issues_df = pd.read_parquet(f"{bundle_dir}/issues_{source}.parquet")
    issues = pd.DatetimeIndex(issues_df["issue_time_utc"])
    print(f"\n--- Running Zero-Shot Inference for {source} ({len(issues)} issues, capacity {cap} MW) ---")
    
    t0 = time.time()
    pred = fc.predict_issues(ds, source, issues, cap, horizon_h=horizon_h)
    wall_sec = round(time.time() - t0, 1)
    
    # Write both file name conventions for compatibility
    pred.to_parquet(f"{out_dir}/{source}_chronos2_zs.parquet", index=False)
    pred.to_parquet(f"{out_dir}/chronos2_zs_{source}.parquet", index=False)
    
    print(f"{source} completed in {wall_sec}s: shape {pred.shape}")
    out_meta["runs"][source] = {
        "issues": len(issues),
        "rows": len(pred),
        "seconds": wall_sec,
        "q50_min": float(pred["q50"].min()),
        "q50_max": float(pred["q50"].max())
    }

with open(f"{out_dir}/meta.json", "w") as f:
    json.dump(out_meta, f, indent=2)

print("\n=== Inferences and meta.json successfully written ===")
